# Convergence-certified positive-MCP event rates in the v16 trajectory coordinates

The notebooks use the exact incoming-ray convention from the original v16 single-point workflow:

$$
\hat{\mathbf u}=(\cos\theta,\,\sin\theta\cos\alpha,\,\sin\theta\sin\alpha),
$$

$$
\hat{\mathbf b}=\cos\psi\,\hat{\mathbf e}_\theta+\sin\psi\,\hat{\mathbf e}_\alpha,
\qquad
\mathbf r_{\mathrm{in}}=b\hat{\mathbf b}-\sqrt{R^2-b^2}\,\hat{\mathbf u},
\qquad
\mathbf v_{\mathrm{in}}=v\hat{\mathbf u}.
$$

The grid now uses the **same validated coupled-Mathieu/Floquet/House pseudopotential-validity classifier in both notebooks**. The local lowest-order $q_{\max}$ and secular/RF values are retained only as diagnostics. They no longer decide whether a grid point is sampled.

The rate outputs are

$$
\dot N_{\mathrm{ph},k},\qquad
\Gamma_{\ge1,k},\qquad
\Gamma_{M,k},
$$

where $M$ is set by `exact_phonon_number` (default $M=1$). The first counts expected phonons per second, while the two $\Gamma$ quantities count passage events per second under a coherent-state/Poisson phonon-number model.

## Notebook B - full staged trajectory calculation

For trap-sensitive rays the solver propagates the MCP in the full v16 DC + RF pseudopotential to the handoff radius, propagates the coupled ion-MCP core, then propagates outward while accumulating the Fourier force history. The analytic adiabatic/Rutherford branches remain explicit shortcuts. Copper hits use the weighted reflection/prompt/delayed branch model.

**Crucially, the expensive staged solver is never called for a grid point outside the validated pseudopotential-valid region.**


The upper copper geometry is now the user-supplied rectangular copper block **minus a rectangular through-gap**. The gap is vacuum and is honored by both screening and staged structure intersections.


In [ ]:
from pathlib import Path
import sys, json, math, time, importlib.util
from dataclasses import fields
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib.colors import LogNorm, TwoSlopeNorm, ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

ROOT = Path.cwd()
# Prefer the versioned convergence-certified module. This avoids accidentally
# loading an older mcp_v16_event_rate.py left in the working directory.
VERSIONED_MODULE = ROOT / "mcp_v16_event_rate_convergence.py"
LEGACY_MODULE = ROOT / "mcp_v16_event_rate.py"
if not VERSIONED_MODULE.exists() and not LEGACY_MODULE.exists():
    ROOT = Path("/mnt/data/mcp_event_rate_v16_compare")
    VERSIONED_MODULE = ROOT / "mcp_v16_event_rate_convergence.py"
    LEGACY_MODULE = ROOT / "mcp_v16_event_rate.py"

MODULE_PATH = (VERSIONED_MODULE if VERSIONED_MODULE.exists() else LEGACY_MODULE).resolve()
if not MODULE_PATH.exists():
    raise FileNotFoundError(
        "Could not find the shared event-rate module beside the notebook. "
        "Keep mcp_v16_event_rate_convergence.py (preferred) or "
        "mcp_v16_event_rate.py in the same folder as this notebook."
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Always create a fresh module object from this exact path; no sys.path/cache import.
spec = importlib.util.spec_from_file_location("mcp_v16_event_rate_active", MODULE_PATH)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not load {MODULE_PATH}")
mcp = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = mcp
spec.loader.exec_module(mcp)

required_functions = [
    "mathieu_validity_mesh", "dense_validity_grid", "mathieu_reference_boundaries",
    "valid_parameter_points", "run_converged_valid_grid", "convergence_audit_point",
    "scanconfig_audit", "hessian_convergence_report", "lower_hoa_quantum_geometry",
    "density_for_point", "ray_ground_first_entry", "add_result_classifications",
    "add_numerical_failure_diagnostics"
]

required_scan_fields = {
    "adaptive_sampling", "min_samples_per_point", "target_mc_rel_se",
    "adaptive_check_every", "exact_phonon_number",
    "density_model", "ode_method", "ode_atol_position_m",
    "ode_atol_velocity_m_s", "ode_atol_quadrature_N_s", "ode_segment_walltime_s",
}
scan_fields = {f.name for f in fields(mcp.ScanConfig)}
missing_fields = sorted(required_scan_fields - scan_fields)
missing_functions = [name for name in required_functions if not hasattr(mcp, name)]
module_version = getattr(mcp, "MODULE_VERSION", "<missing>")
if missing_fields or missing_functions or module_version < "2026-08-13.2":
    raise ImportError(
        "The shared event-rate module is outdated or mismatched.\n"
        f"Loaded: {MODULE_PATH}\n"
        f"MODULE_VERSION={module_version}\n"
        f"Missing ScanConfig fields={missing_fields}\n"
        f"Missing functions={missing_functions}\n"
        "Replace the module with mcp_v16_event_rate_convergence.py supplied "
        "with this notebook, restart the kernel, and run from the first cell."
    )

ScanConfig = mcp.ScanConfig
PRXPopulationConfig = mcp.PRXPopulationConfig
CopperGroundPlane = mcp.CopperGroundPlane
direction_basis = mcp.direction_basis
initial_state_from_v16 = mcp.initial_state_from_v16
mathieu_metrics = mcp.mathieu_metrics
mathieu_validity_mesh = mcp.mathieu_validity_mesh
dense_validity_grid = mcp.dense_validity_grid
mathieu_reference_boundaries = mcp.mathieu_reference_boundaries
run_grid = mcp.run_grid
result_matrix = mcp.result_matrix
compare_frames = mcp.compare_frames
save_config = mcp.save_config
parameter_grid = mcp.parameter_grid

scanconfig_audit = mcp.scanconfig_audit
hessian_convergence_report = mcp.hessian_convergence_report
lower_hoa_quantum_geometry = mcp.lower_hoa_quantum_geometry
density_for_point = mcp.density_for_point
ConvergenceConfig = mcp.ConvergenceConfig
valid_parameter_points = mcp.valid_parameter_points
run_converged_valid_grid = mcp.run_converged_valid_grid
convergence_audit_point = mcp.convergence_audit_point
add_result_classifications = mcp.add_result_classifications
add_numerical_failure_diagnostics = mcp.add_numerical_failure_diagnostics
RESULT_CLASS_LABELS = mcp.RESULT_CLASS_LABELS
NUMERICAL_FAILURE_LABELS = mcp.NUMERICAL_FAILURE_LABELS

OUTPUT = ROOT / "outputs"
OUTPUT.mkdir(exist_ok=True)
print("Using module:", MODULE_PATH)
print("Module version:", mcp.MODULE_VERSION)
print("Validated v16 coupled-Floquet/House mask is active.")

staged_single_sample = mcp.staged_single_sample
classify_collision_regime = mcp.classify_collision_regime
certify_radii = mcp.certify_radii


## 1. Configuration and convergence profiles

The default `reliable` profile uses the same $36\times36$ mass-charge grid as the screening notebook, but **the trajectory solver is called only for grid points inside the validated v16 pseudopotential-valid region**. Invalid points are never phase-space sampled.

For every valid $(m_\chi,\epsilon)$ point the notebook estimates the central rates and then performs secondary convergence audits when a point is accurate enough to be a candidate for the `well_converged` or `precision_certified` classes. The expensive secondary audit is skipped when every observable already has MC relative error above 10%, because such a point cannot be promoted above the `estimated` class in the current run.

The output no longer reduces every point to a single pass/fail flag. Each observable is assigned one of six quality classes:

- **precision_certified**: MC relative SE $\le5\%$ and the $b$-support, numerical-resolution, and timeout audits all pass;
- **well_converged**: MC relative SE $\le10\%$ and the same secondary audits pass;
- **estimated**: finite estimate with MC relative SE $\le20\%$ **and bounded impact-parameter support**, but not fully certified;
- **support_unresolved**: a finite central estimate exists, but the adaptive $b$ support has not been bounded to the requested tolerance;
- **noisy**: bounded support but MC relative SE between 20% and 50%;
- **unresolved**: non-finite result, excessive trajectory timeouts, worker failure, or MC relative SE $>50\%$.

Version 2026-08-13.2 also distinguishes a statistically ambiguous tail from a demonstrably long tail.  When the 2-sigma tail bound fails but its lower confidence bound is still compatible with the requested tolerance, the code increases the **annulus sample count at the same physical support** before enlarging $b_{\max}$. Only when the tail is demonstrably too large is the physical support expanded. Old expanded-support results and their final annulus summaries are reused, so a previous $32B$ point continues from $32B$ rather than restarting at $B$.

The strict historical `reliable_*` flags are retained and correspond to the strongest `precision_certified` class. The central estimates are **not discarded** merely because they fail 5% certification; use the rate, uncertainty, and classification maps together.

### ODE tolerance convention

The staged solver uses adaptive `scipy.integrate.solve_ivp` (default method `DOP853`). There is **no fixed $dt$**. `ode_max_step_s` is only an upper bound on the internal adaptive step. For each state component $y_i$, the local error scale is $\mathrm{atol}_i+\mathrm{rtol}|y_i|$. Separate absolute tolerances are used for positions, velocities, and Fourier-impulse quadratures because those state components have different physical units.


In [ ]:
RUN_PROFILE = "reliable"  # "smoke", "balanced", or "reliable"

COMMON = dict(
    m_min_kg=1e-30, m_max_kg=1e-16,
    eps_min=1e-8, eps_max=1.0,
    density_model="constant_local", density_m3=1e9, temperature_K=300.0,
    target_mode=2, exact_phonon_number=1,
    seed=20260805,
)

if RUN_PROFILE == "smoke":
    cfg = ScanConfig(
        **COMMON, n_mass=18, n_eps=18,
        samples_per_point=16, max_ode_samples_per_point=16,
        adaptive_sampling=False,
        b_cap_m=2e-4, R_outer_m=20e-3,
        ode_method="DOP853", ode_rtol=2e-7,
        ode_atol_position_m=2e-9, ode_atol_velocity_m_s=2e-6,
        ode_atol_quadrature_N_s=2e-28, ode_max_step_s=2e-6,
        ode_segment_walltime_s=10.0, max_stage_time_s=1e-2, max_ground_interactions=2,
        delayed_branch_samples=2,
    )
    conv = ConvergenceConfig(
        mc_target_rel_se=0.50, mc_min_samples=4, mc_max_samples=8, mc_check_every=2,
        diagnostic_samples=2, b_tail_samples=2,
        b_tail_fraction_tol=1.0, adaptive_bmax_enabled=True, adaptive_bmax_max_expansions=0, adaptive_bmax_max_factor=1.0,
        adaptive_bmax_tail_max_samples=8, adaptive_bmax_tail_sample_growth=2.0, adaptive_bmax_version=2,
        numerical_rel_tol=1.0, confidence_z=1.0, max_timeout_fraction=1.0,
        precision_rel_se=0.05, well_converged_rel_se=0.10, estimated_rel_se=0.20, noisy_rel_se=0.50,
    )
elif RUN_PROFILE == "balanced":
    cfg = ScanConfig(
        **COMMON, n_mass=36, n_eps=36,
        samples_per_point=256, max_ode_samples_per_point=256,
        adaptive_sampling=False,
        b_cap_m=1e-3, R_outer_m=40e-3,
        ode_method="DOP853", ode_rtol=1e-8,
        ode_atol_position_m=1e-10, ode_atol_velocity_m_s=1e-7,
        ode_atol_quadrature_N_s=1e-30, ode_max_step_s=1e-7,
        ode_segment_walltime_s=20.0, max_stage_time_s=0.20, max_ground_interactions=4,
        delayed_branch_samples=4,
    )
    conv = ConvergenceConfig(
        mc_target_rel_se=0.08, mc_min_samples=64, mc_max_samples=256, mc_check_every=16,
        diagnostic_samples=24, b_tail_samples=32,
        b_tail_fraction_tol=0.10, adaptive_bmax_enabled=True, adaptive_bmax_max_expansions=6, adaptive_bmax_max_factor=64.0,
        adaptive_bmax_tail_max_samples=256, adaptive_bmax_tail_sample_growth=2.0, adaptive_bmax_version=2,
        numerical_rel_tol=0.10, confidence_z=2.0, max_timeout_fraction=0.02,
        precision_rel_se=0.05, well_converged_rel_se=0.10, estimated_rel_se=0.20, noisy_rel_se=0.50,
    )
elif RUN_PROFILE == "reliable":
    cfg = ScanConfig(
        **COMMON, n_mass=36, n_eps=36,
        samples_per_point=2048, max_ode_samples_per_point=2048,
        adaptive_sampling=False,
        b_cap_m=1e-3, R_outer_m=40e-3,
        ode_method="DOP853", ode_rtol=3e-9,
        ode_atol_position_m=3e-11, ode_atol_velocity_m_s=3e-8,
        ode_atol_quadrature_N_s=3e-31, ode_max_step_s=5e-8,
        ode_segment_walltime_s=30.0, max_stage_time_s=0.30, max_ground_interactions=6,
        delayed_branch_samples=8,
    )
    conv = ConvergenceConfig(
        mc_target_rel_se=0.05, mc_min_samples=256, mc_max_samples=2048, mc_check_every=64,
        diagnostic_samples=64, b_tail_samples=128,
        b_tail_factor_1=2.0, b_tail_factor_2=4.0,
        b_tail_fraction_tol=0.05, b_tail_decay_factor=1.25,
        adaptive_bmax_enabled=True, adaptive_bmax_max_expansions=8, adaptive_bmax_max_factor=256.0,
        adaptive_bmax_tail_max_samples=1024, adaptive_bmax_tail_sample_growth=2.0, adaptive_bmax_version=2,
        numerical_rel_tol=0.05, confidence_z=2.0,
        outer_radius_factor=1.5, r_far_factor=2.0,
        timestep_factor=0.5, stage_time_factor=2.0, r_switch_factor=1.5,
        delayed_samples_factor=2, extra_ground_interactions=2,
        max_timeout_fraction=0.01,
        precision_rel_se=0.05, well_converged_rel_se=0.10, estimated_rel_se=0.20, noisy_rel_se=0.50,
    )
else:
    raise ValueError(RUN_PROFILE)

ground = CopperGroundPlane(
    # User-supplied CSG geometry defaults:
    # outer block [(-12063,-10759,1376),(13413,10759,2977)] um
    # minus through-gap [(-4115,-1651,1376),(2542,1651,2977)] um
    double_layer_eV=3.19,
)

save_config(cfg, ground, OUTPUT / "staged_config.json")
(OUTPUT / "staged_convergence_config.json").write_text(json.dumps(conv.__dict__, indent=2))

SHOW_LIVE_PROGRESS = True

# Production parallelism. These are independent CPU worker PROCESSES rather
# than Python ThreadPool threads: this avoids the GIL and lets the watchdog
# terminate one pathological solve_ivp point without freezing the notebook.
PARALLEL_WORKERS = 8
PARALLEL_BACKEND = "subprocess"
POINT_WALLTIME_S = 30 * 60.0   # hard complete-point watchdog (30 min)
HEARTBEAT_S = 30.0             # notebook prints running-worker status every 30 s

# Keep the same checkpoint filename so a current compatible run can resume.
CONVERGENCE_CHECKPOINT = OUTPUT / f"staged_convergence_valid_partial_{RUN_PROFILE}.csv"
print(cfg)
print(conv)
print("Checkpoint:", CONVERGENCE_CHECKPOINT)
print("Parallel point workers:", PARALLEL_WORKERS)
print("Parallel backend:", PARALLEL_BACKEND)
print("Per-point wall-clock watchdog [s]:", POINT_WALLTIME_S)
print("Per-solve_ivp segment watchdog [s]:", cfg.ode_segment_walltime_s)


### Configuration and geometry audit helpers
These tables/dictionaries are generated by the shared module and mirror the corresponding sections of the documentation PDF.


In [ ]:
display(scanconfig_audit())
print("Upper copper outer bounds [m]:", ground.lo, ground.hi)
print("Upper copper gap bounds [m]:", ground.gap_lo, ground.gap_hi)
print("Lower HOA field-geometry audit:")
lower_hoa_quantum_geometry()


## 2. Validity preview and strict skip guarantee

The validated v16 coupled-Floquet/House mask is evaluated before any trajectory sampling. The convergence runner constructs a list containing **only** valid grid points and iterates over that list. Therefore the expensive staged solver never receives a pseudopotential-invalid $(m_\chi,\epsilon)$ pair.

The background plot still displays stable-but-pseudopotential-invalid and unstable regions so the accepted domain can be audited visually.


In [ ]:
def _log_edges(values):
    values = np.asarray(values, float)
    lv = np.log(values)
    edges = np.empty(values.size + 1)
    edges[1:-1] = np.exp(0.5 * (lv[:-1] + lv[1:]))
    edges[0] = np.exp(lv[0] - 0.5 * (lv[1] - lv[0]))
    edges[-1] = np.exp(lv[-1] + 0.5 * (lv[-1] - lv[-2]))
    return edges


def _full_grid_matrix(df, cfg, column):
    """Map rows onto the exact configured grid using stable log-space keys.

    Do not rely on exact float equality after CSV round trips.  The convergence
    runner now returns one canonical row per valid point, but this mapping also
    makes plots robust when inspecting older checkpoint files.
    """
    masses, eps = parameter_grid(cfg)
    Z = np.full((len(masses), len(eps)), np.nan, dtype=float)
    if column not in df.columns or df.empty:
        return masses, eps, Z
    mi = {round(math.log10(float(v)), 9): i for i, v in enumerate(masses)}
    ei = {round(math.log10(float(v)), 9): j for j, v in enumerate(eps)}
    for _, row in df.iterrows():
        try:
            m = float(row["m_dm_kg"]); e = float(row["eps"])
            i = mi.get(round(math.log10(m), 9)); j = ei.get(round(math.log10(e), 9))
            value = float(row[column])
        except (KeyError, TypeError, ValueError):
            continue
        if i is not None and j is not None:
            Z[i, j] = value
    return masses, eps, Z


def _valid_grid_mask(cfg):
    masses, eps = parameter_grid(cfg)
    V = np.zeros((len(masses), len(eps)), dtype=bool)
    mi = {round(math.log10(float(v)), 9): i for i, v in enumerate(masses)}
    ei = {round(math.log10(float(v)), 9): j for j, v in enumerate(eps)}
    for rec in valid_parameter_points(cfg).to_dict("records"):
        i = mi[round(math.log10(float(rec["m_dm_kg"])), 9)]
        j = ei[round(math.log10(float(rec["eps"])), 9)]
        V[i, j] = True
    return masses, eps, V


def _overlay_missing_or_zero(ax, cfg, Z, *, show_zero=False):
    masses, eps, V = _valid_grid_mask(cfg)
    missing = V & ~np.isfinite(Z)
    if np.any(missing):
        ii, jj = np.where(missing)
        ax.scatter(eps[jj], masses[ii], marker="x", s=34, linewidths=1.0,
                   color="black", zorder=8, label="valid point without finite central estimate")
    if show_zero:
        zeros = V & np.isfinite(Z) & (Z == 0.0)
        if np.any(zeros):
            ii, jj = np.where(zeros)
            ax.scatter(eps[jj], masses[ii], marker="o", s=30, facecolors="none",
                       edgecolors="black", linewidths=0.9, zorder=8, label="finite zero estimate")


def _draw_reference_background(ax, cfg):
    md, ed, stable, pseudo, _, _ = dense_validity_grid(cfg, n_mass=700, n_eps=700)
    classes = np.where(~stable, 0, np.where(pseudo, 2, 1)).T
    bg_cmap = ListedColormap(["white", "0.78", "0.34"])
    bg_norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], bg_cmap.N)
    ax.pcolormesh(_log_edges(ed), _log_edges(md), classes, shading="flat",
                  cmap=bg_cmap, norm=bg_norm, zorder=0)
    e_line = np.geomspace(cfg.eps_min, cfg.eps_max, 1800)
    for boundary in mathieu_reference_boundaries(cfg):
        m_line = boundary["slope_kg"] * e_line
        visible = (m_line >= cfg.m_min_kg) & (m_line <= cfg.m_max_kg)
        if not np.any(visible):
            continue
        kind = boundary["kind"]
        linestyle = {"pseudopotential": ":", "stability": "--",
                     "stability+pseudopotential": "-."}[kind]
        lw = 0.9 if kind == "pseudopotential" else 1.35
        ax.plot(e_line[visible], m_line[visible], color="black",
                linestyle=linestyle, linewidth=lw, alpha=0.95, zorder=7)


def _validity_legend():
    return [
        Patch(facecolor="0.34", edgecolor="none", label="Stable; pseudopotential valid"),
        Patch(facecolor="0.78", edgecolor="none", label="Stable; pseudopotential not valid"),
        Patch(facecolor="white", edgecolor="0.4", label="Unstable"),
        Line2D([0], [0], color="black", linestyle=":", label="Pseudopotential boundary"),
        Line2D([0], [0], color="black", linestyle="--", label="Mathieu stability boundary"),
        Line2D([0], [0], color="black", linestyle="-.", label="Combined boundary"),
    ]


def _format_axes(ax, cfg, title):
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlim(cfg.eps_min, cfg.eps_max); ax.set_ylim(cfg.m_min_kg, cfg.m_max_kg)
    ax.set_xlabel(r"Charge fraction $\epsilon$ ($Q=\epsilon e$)")
    ax.set_ylabel(r"DM mass $m_\chi$ [kg]")
    ax.set_title(title)


def plot_validity_reference(cfg, path):
    fig, ax = plt.subplots(figsize=(9.5, 6.7))
    _draw_reference_background(ax, cfg)
    ax.legend(handles=_validity_legend(), loc="lower right", fontsize=8, framealpha=0.9)
    _format_axes(ax, cfg, "Coupled Mathieu stability and pseudopotential validity\nvalidated v16 boundary topology")
    fig.tight_layout(); fig.savefig(path, dpi=180); plt.show()


def plot_metric(df, cfg, column, title, colorbar_label, path, contours=True):
    masses, eps, Z = _full_grid_matrix(df, cfg, column)
    finite = np.isfinite(Z) & (Z > 0)
    fig, ax = plt.subplots(figsize=(9.5, 6.7))
    _draw_reference_background(ax, cfg)
    if finite.any():
        P = np.ma.masked_where(~finite, Z)
        vals = Z[finite]
        vmin, vmax = float(np.nanmin(vals)), float(np.nanmax(vals))
        if vmax <= vmin * (1 + 1e-12):
            vmin /= np.sqrt(10.0); vmax *= np.sqrt(10.0)
        cmap = plt.get_cmap("viridis").copy(); cmap.set_bad((0, 0, 0, 0))
        im = ax.pcolormesh(_log_edges(eps), _log_edges(masses), P, shading="flat",
                           cmap=cmap, norm=LogNorm(vmin=vmin, vmax=vmax), zorder=3)
        fig.colorbar(im, ax=ax).set_label(colorbar_label)
        if contours and np.count_nonzero(finite) >= 10:
            L = np.log10(np.where(finite, Z, np.nan))
            lo, hi = np.nanmin(L), np.nanmax(L)
            if hi - lo > 0.4:
                levels = np.unique(np.round(np.linspace(lo, hi, 5), 1))
                cs = ax.contour(eps, masses, L, levels=levels, colors="black",
                                linewidths=0.75, zorder=6)
                ax.clabel(cs, fmt=lambda x: f"$10^{{{x:.1f}}}$", fontsize=8)
    _overlay_missing_or_zero(ax, cfg, Z, show_zero=True)
    handles = _validity_legend()[1:]
    extra_handles, extra_labels = ax.get_legend_handles_labels()
    ax.legend(handles=handles + extra_handles, loc="lower right", fontsize=8, framealpha=0.88)
    _format_axes(ax, cfg, title)
    fig.tight_layout(); fig.savefig(path, dpi=180); plt.show()


def plot_convergence_quantity(df, cfg, column, title, label, path, vmax=None):
    masses, eps, Z = _full_grid_matrix(df, cfg, column)
    finite = np.isfinite(Z) & (Z >= 0)
    fig, ax = plt.subplots(figsize=(9.5, 6.7))
    _draw_reference_background(ax, cfg)
    if finite.any():
        P = np.ma.masked_where(~finite, Z)
        kwargs = {}
        if vmax is not None:
            kwargs.update(vmin=0.0, vmax=vmax)
        im = ax.pcolormesh(_log_edges(eps), _log_edges(masses), P, shading="flat",
                           cmap="viridis", zorder=3, **kwargs)
        fig.colorbar(im, ax=ax).set_label(label)
    _format_axes(ax, cfg, title)
    fig.tight_layout(); fig.savefig(path, dpi=180); plt.show()


def plot_reliability(df, cfg, flag, title, path):
    masses, eps, Z = _full_grid_matrix(df, cfg, flag)
    fig, ax = plt.subplots(figsize=(9.5, 6.7))
    _draw_reference_background(ax, cfg)
    finite = np.isfinite(Z)
    if finite.any():
        P = np.ma.masked_where(~finite, Z.astype(float))
        cmap = ListedColormap(["0.85", "tab:green"])
        norm = BoundaryNorm([-0.5, 0.5, 1.5], cmap.N)
        ax.pcolormesh(_log_edges(eps), _log_edges(masses), P, shading="flat",
                      cmap=cmap, norm=norm, zorder=3)
    ax.legend(handles=[Patch(facecolor="tab:green", label="passes all tests"),
                       Patch(facecolor="0.85", label="fails >=1 required test")],
              loc="lower right", fontsize=8)
    _format_axes(ax, cfg, title)
    fig.tight_layout(); fig.savefig(path, dpi=180); plt.show()


def plot_relative_error(df, cfg, column, title, path, cap_pct=50.0):
    masses, eps, Z = _full_grid_matrix(df, cfg, column)
    Zpct = 100.0 * Z
    finite = np.isfinite(Zpct) & (Zpct >= 0)
    fig, ax = plt.subplots(figsize=(9.5, 6.7))
    _draw_reference_background(ax, cfg)
    if finite.any():
        # Saturate at the unresolved/noisy boundary so the useful 0-50%
        # structure remains visible; exact uncapped values remain in the CSV.
        P = np.ma.masked_where(~finite, np.minimum(Zpct, cap_pct))
        im = ax.pcolormesh(_log_edges(eps), _log_edges(masses), P, shading="flat",
                           cmap="viridis", vmin=0.0, vmax=cap_pct, zorder=3)
        fig.colorbar(im, ax=ax).set_label(
            f"Monte Carlo relative standard error [%] (>= {cap_pct:g}% saturated)"
        )
    _overlay_missing_or_zero(ax, cfg, Z, show_zero=False)
    _format_axes(ax, cfg, title)
    fig.tight_layout(); fig.savefig(path, dpi=180); plt.show()


def plot_classification(df, cfg, code_column, title, path):
    masses, eps, Z = _full_grid_matrix(df, cfg, code_column)
    finite = np.isfinite(Z)
    fig, ax = plt.subplots(figsize=(9.5, 6.7))
    _draw_reference_background(ax, cfg)
    if finite.any():
        P = np.ma.masked_where(~finite, Z.astype(float))
        # 0 unresolved, 1 noisy, 2 support_unresolved, 3 estimated,
        # 4 well_converged, 5 precision_certified.
        cmap = ListedColormap(["0.30", "tab:red", "tab:purple", "tab:orange", "tab:blue", "tab:green"])
        norm = BoundaryNorm(np.arange(-0.5, 6.5, 1.0), cmap.N)
        ax.pcolormesh(_log_edges(eps), _log_edges(masses), P, shading="flat",
                      cmap=cmap, norm=norm, zorder=3)
    _overlay_missing_or_zero(ax, cfg, Z, show_zero=False)
    handles = [
        Patch(facecolor="tab:green", label="precision certified (<=5% + support + numerical audits)"),
        Patch(facecolor="tab:blue", label="well converged (<=10% + support + numerical audits)"),
        Patch(facecolor="tab:orange", label="estimated (<=20%, support bounded)"),
        Patch(facecolor="tab:purple", label="support unresolved (finite rate, b tail open)"),
        Patch(facecolor="tab:red", label="noisy (20-50%, support bounded)"),
        Patch(facecolor="0.30", label="unresolved (>50% / failure / nonfinite)"),
    ]
    ax.legend(handles=handles, loc="lower right", fontsize=7.2, framealpha=0.92)
    _format_axes(ax, cfg, title)
    fig.tight_layout(); fig.savefig(path, dpi=180); plt.show()

def plot_precision_certified_metric(df, cfg, rate_column, class_code_column, title, colorbar_label, path):
    """Show certified rates without mislabeling finite non-certified points as missing."""
    masses, eps, R = _full_grid_matrix(df, cfg, rate_column)
    _, _, C = _full_grid_matrix(df, cfg, class_code_column)
    _, _, V = _valid_grid_mask(cfg)
    finite_rate = V & np.isfinite(R)
    certified = finite_rate & np.isfinite(C) & (C == 5)
    noncert = finite_rate & ~certified
    missing = V & ~finite_rate
    fig, ax = plt.subplots(figsize=(9.5, 6.7))
    _draw_reference_background(ax, cfg)
    if np.any(noncert):
        status = np.ma.masked_where(~noncert, np.ones_like(R))
        ax.pcolormesh(_log_edges(eps), _log_edges(masses), status, shading="flat",
                      cmap=ListedColormap(["0.72"]), vmin=0.5, vmax=1.5, zorder=2.5)
    if np.any(certified):
        P = np.ma.masked_where(~certified, R)
        vals = R[certified]
        vmin, vmax = float(np.nanmin(vals)), float(np.nanmax(vals))
        if vmax <= vmin * (1 + 1e-12): vmin /= np.sqrt(10.0); vmax *= np.sqrt(10.0)
        im = ax.pcolormesh(_log_edges(eps), _log_edges(masses), P, shading="flat",
                           cmap="viridis", norm=LogNorm(vmin=vmin, vmax=vmax), zorder=3)
        fig.colorbar(im, ax=ax).set_label(colorbar_label)
    if np.any(missing):
        ii, jj = np.where(missing)
        ax.scatter(eps[jj], masses[ii], marker="x", s=34, linewidths=1.0, color="black", zorder=8)
    handles = _validity_legend()[1:] + [
        Patch(facecolor="0.72", edgecolor="none", label="finite central estimate; not precision certified"),
        Line2D([0], [0], color="black", marker="x", linestyle="None", label="valid point without finite central estimate"),
    ]
    ax.legend(handles=handles, loc="lower right", fontsize=7.5, framealpha=0.9)
    _format_axes(ax, cfg, title)
    fig.tight_layout(); fig.savefig(path, dpi=180); plt.show()

def plot_numerical_failure_mode(df, cfg, code_column, title, path):
    """Show a numerical limiter, or explicitly why the numerical audit was not run."""
    masses, eps, Z = _full_grid_matrix(df, cfg, code_column)
    finite = np.isfinite(Z)
    fig, ax = plt.subplots(figsize=(9.5, 6.7))
    _draw_reference_background(ax, cfg)
    if finite.any():
        P = np.ma.masked_where(~finite, Z.astype(float))
        cmap = ListedColormap([
            "0.82", "0.35", "tab:blue", "tab:orange", "tab:green",
            "tab:red", "tab:purple", "tab:brown", "0.15", "tab:pink"
        ])
        norm = BoundaryNorm(np.arange(-0.5, 10.5, 1.0), cmap.N)
        ax.pcolormesh(_log_edges(eps), _log_edges(masses), P, shading="flat", cmap=cmap, norm=norm, zorder=3)
    legend_order = [1,2,3,4,5,6,7,9,0,8]
    facecolors = ["0.35","tab:blue","tab:orange","tab:green","tab:red","tab:purple","tab:brown","tab:pink","0.82","0.15"]
    handles = [Patch(facecolor=c, label=NUMERICAL_FAILURE_LABELS[k]) for k,c in zip(legend_order,facecolors)]
    ax.legend(handles=handles, loc="lower right", fontsize=7.0, framealpha=0.92, ncol=2)
    _format_axes(ax, cfg, title)
    fig.tight_layout(); fig.savefig(path, dpi=180); plt.show()

def plot_support_status(df, cfg, path):
    status_to_code = {
        "converged": 0,
        "tail_noise_limited": 1,
        "support_unresolved_max_factor": 2,
        "geometry_limited": 3,
        "in_progress": 4,
    }
    tmp = df.copy()
    tmp["_support_status_code"] = [status_to_code.get(str(x), 5) for x in tmp.get("adaptive_bmax_status", "")]
    masses, eps, Z = _full_grid_matrix(tmp, cfg, "_support_status_code")
    finite=np.isfinite(Z)
    fig,ax=plt.subplots(figsize=(9.5,6.7)); _draw_reference_background(ax,cfg)
    cmap=ListedColormap(["tab:green","tab:orange","tab:red","tab:purple","tab:blue","0.35"])
    norm=BoundaryNorm(np.arange(-0.5,6.5,1),cmap.N)
    if finite.any():
        ax.pcolormesh(_log_edges(eps),_log_edges(masses),np.ma.masked_where(~finite,Z),shading="flat",cmap=cmap,norm=norm,zorder=3)
    labels=["converged","tail_noise_limited","max_factor_reached","geometry_limited","in_progress","other/not audited"]
    ax.legend(handles=[Patch(facecolor=cmap.colors[i],label=labels[i]) for i in range(6)],loc="lower right",fontsize=7.2,framealpha=0.92)
    _format_axes(ax,cfg,"Adaptive impact-parameter support status")
    fig.tight_layout(); fig.savefig(path,dpi=180); plt.show()


def flatten_adaptive_b_history(df):
    """Create a reusable per-support diagnostic table from checkpoint JSON history."""
    rows=[]
    for _,r in df.iterrows():
        try: hist=json.loads(str(r.get("adaptive_bmax_history","[]")))
        except Exception: hist=[]
        if not isinstance(hist,list): continue
        for j,h in enumerate(hist):
            if not isinstance(h,dict): continue
            rec={"m_dm_kg":r.get("m_dm_kg"),"eps":r.get("eps"),"history_index":j,
                 "support_factor":h.get("support_factor",np.nan),"annulus_n":h.get("annulus_n",np.nan),
                 "decision":h.get("decision",h.get("source","legacy")),"clipped":h.get("clipped",False)}
            for lab in ("phonon","ge1","exact_M"):
                mm=h.get("metrics",{}).get(lab,{}) if isinstance(h.get("metrics",{}),dict) else {}
                for key in ("ann1_mean","ann1_se","ann2_mean","ann2_se","frac_mean","frac_lower","frac_upper"):
                    rec[f"{lab}_{key}"]=mm.get(key,np.nan)
            rows.append(rec)
    return pd.DataFrame(rows)


def plot_tail_history_representatives(history_df, audit_df, path, n_points=6):
    """Plot tail-fraction evolution for the most demanding support-repair points."""
    if history_df.empty: return
    candidates=audit_df.copy()
    candidates["_tail"] = pd.to_numeric(candidates.get("b_tail_phonon_fraction_upper"),errors="coerce")
    candidates["_factor"] = pd.to_numeric(candidates.get("adaptive_bmax_factor"),errors="coerce")
    candidates=candidates[np.isfinite(candidates["_factor"])].sort_values(["_factor","_tail"],ascending=False).head(n_points)
    if candidates.empty: return
    fig,ax=plt.subplots(figsize=(9.5,6.2))
    for _,r in candidates.iterrows():
        sel=history_df[np.isclose(history_df.m_dm_kg,float(r.m_dm_kg),rtol=1e-10)&np.isclose(history_df.eps,float(r.eps),rtol=1e-10)].copy()
        sel=sel[np.isfinite(pd.to_numeric(sel["phonon_frac_upper"],errors="coerce"))]
        if sel.empty: continue
        sel=sel.sort_values(["support_factor","annulus_n"])
        ax.plot(sel.support_factor,sel.phonon_frac_upper,marker="o",label=rf"$m={float(r.m_dm_kg):.1e},\ \epsilon={float(r.eps):.1e}$")
    ax.axhline(conv.b_tail_fraction_tol,linestyle="--",linewidth=1.0,label="tail tolerance")
    ax.set_xscale("log",base=2); ax.set_yscale("log")
    ax.set_xlabel(r"central support factor $B_{\rm final}/B_{\rm nominal}$")
    ax.set_ylabel(r"phonon tail 2-sigma upper fraction")
    ax.set_title("Adaptive-b tail history for representative difficult points")
    ax.legend(fontsize=7.0,framealpha=0.9)
    fig.tight_layout(); fig.savefig(path,dpi=180); plt.show()


def plot_observable_ratio(df, cfg, numerator, denominator, title, colorbar_label, path):
    masses, eps, N = _full_grid_matrix(df, cfg, numerator)
    _, _, D = _full_grid_matrix(df, cfg, denominator)
    _, _, V = _valid_grid_mask(cfg)
    ratio = np.full_like(N, np.nan, dtype=float)
    finite_inputs = V & np.isfinite(N) & np.isfinite(D)
    good = finite_inputs & (D > 0.0) & (N >= 0.0)
    ratio[good] = N[good] / D[good]
    fig, ax = plt.subplots(figsize=(9.5, 6.7))
    _draw_reference_background(ax, cfg)
    if np.any(good):
        P = np.ma.masked_where(~good, np.minimum(ratio, 1.0))
        im = ax.pcolormesh(_log_edges(eps), _log_edges(masses), P, shading="flat",
                           cmap="viridis", vmin=0.0, vmax=1.0, zorder=3)
        fig.colorbar(im, ax=ax).set_label(colorbar_label)
    missing = V & ~finite_inputs
    if np.any(missing):
        ii, jj = np.where(missing)
        ax.scatter(eps[jj], masses[ii], marker="x", s=34, linewidths=1.0,
                   color="black", zorder=8, label="missing central input")
    zero_den = finite_inputs & (D == 0.0)
    if np.any(zero_den):
        ii, jj = np.where(zero_den)
        ax.scatter(eps[jj], masses[ii], marker="o", s=30, facecolors="none",
                   edgecolors="black", linewidths=0.9, zorder=8, label="ratio undefined: denominator = 0")
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(handles=handles, labels=labels, loc="lower right", fontsize=8, framealpha=0.9)
    _format_axes(ax, cfg, title)
    fig.tight_layout(); fig.savefig(path, dpi=180); plt.show()


plot_validity_reference(cfg, OUTPUT / "mathieu_validity_reference.png")
valid_points = valid_parameter_points(cfg)
print(f"Full grid: {cfg.n_mass * cfg.n_eps} points")
print(f"Validated pseudopotential-valid points: {len(valid_points)}")
print(f"Trajectory solver calls outside valid region: 0 by construction")
display(valid_points.head())


## 3. One staged trajectory diagnostic

Use a parameter point inside the validated low-mass pseudopotential-valid island. This is a software/physics diagnostic, not part of the grid estimator.

In [ ]:
example = dict(m_dm=1e-25, eps=0.1, speed=100.0, b=10e-6,
               theta=1.0, alpha=2.0, psi=0.5)
print("validity:", mathieu_metrics(example["m_dm"], example["eps"], cfg))
print("regime:", classify_collision_regime(example["speed"], example["b"]))
print("radii:", certify_radii(**example, cfg=cfg))
diag = staged_single_sample(**example, cfg=cfg, ground=ground, seed=123)
diag


## 4. Run the convergence-certified staged scan

This is the production driver. It iterates over `valid_parameter_points(cfg)`, not over the rectangular grid. The checkpoint contains only valid points and can be resumed after interruption.

**Adaptive impact-parameter support (v2026-08-13.1):** the Rutherford $B=b_{\max}$ is now only the initial support radius. The code samples the explicit $[B,2B]$ and $[2B,4B]$ annuli. If the tail is too large, the inner annulus is promoted into the central integral, the support becomes $2B$, and only the next new outer annulus is simulated. The process continues geometrically until the tail passes or the configured/geometric limit is reached. Means are added and independent standard errors are combined in quadrature.

**Previous central results are reused.** When the checkpoint already contains a finite $[0,B]$ central estimate from an older notebook, the runner submits a `support_upgrade` job rather than rerunning the 2048-sample central Monte Carlo. Only the adaptive outer-$b$ annuli and any required paired numerical diagnostics are evaluated. New points with no finite central estimate still run the complete calculation.

The adaptive $b$-tail audit is always performed because it can change the central rate itself. The expensive paired numerical-resolution audit is still skipped when all observables remain above the 10% `well_converged` MC threshold.


### Robust 8-worker convergence run

The production runner uses **8 independent Python worker processes** (not GIL-bound Python threads). Each parameter point is isolated, checkpointed by the parent process, and protected by two watchdogs:

- `cfg.ode_segment_walltime_s`: aborts one pathological `solve_ivp` segment.
- `POINT_WALLTIME_S`: terminates an entire parameter-point worker if it exceeds the wall-clock budget, records that point as unreliable, and immediately continues with the remaining valid points.

A heartbeat is printed every `HEARTBEAT_S`, including the most recent line from each worker log. This prevents a long silent interval from looking like a frozen kernel. Only points inside the validated pseudopotential region are submitted. Legacy finite points appear as `support_upgrade` workers; they reuse the previous central estimate. Points lacking a finite central value appear as `central` workers.


In [ ]:
t0 = time.time()
audit_df = run_converged_valid_grid(
    cfg, ground, conv,
    progress=SHOW_LIVE_PROGRESS,
    checkpoint_path=CONVERGENCE_CHECKPOINT,
    resume=True,
    max_workers=PARALLEL_WORKERS,
    parallel_backend=PARALLEL_BACKEND,
    point_timeout_s=POINT_WALLTIME_S,
    heartbeat_s=HEARTBEAT_S,
)
print(f"Total runtime this session: {time.time() - t0:.2f} s")

# Add/refresh the six-level quality classification.  This also upgrades old
# checkpoint rows without forcing completed central estimates to be rerun.
audit_df = add_result_classifications(audit_df, conv)

# Canonical-grid integrity check.  The runner must return exactly one row for
# each CURRENT pseudopotential-valid point; old duplicate/stale checkpoint rows
# are not allowed to inflate the denominator.
_expected_valid = valid_parameter_points(cfg)
_expected_n = len(_expected_valid)
_actual_unique = len({(round(math.log10(float(r.m_dm_kg)), 9), round(math.log10(float(r.eps)), 9)) for _, r in audit_df.iterrows()})
_central_complete = (
    np.isfinite(pd.to_numeric(audit_df["phonon_rate_s"], errors="coerce"))
    & np.isfinite(pd.to_numeric(audit_df["event_rate_ge1_s"], errors="coerce"))
    & np.isfinite(pd.to_numeric(audit_df["event_rate_exact_M_s"], errors="coerce"))
)
print(f"Current-grid integrity: rows={len(audit_df)}, unique={_actual_unique}, expected valid={_expected_n}")
print(f"Finite central estimates: {int(_central_complete.sum())}/{_expected_n}; missing/failed central estimates: {int((~_central_complete).sum())}")
if len(audit_df) != _expected_n or _actual_unique != _expected_n:
    raise RuntimeError("Current-grid table is not one-row-per-valid-point; do not plot this result.")

# Add worst-case numerical shifts and the identity of the limiting numerical test.
audit_df = add_numerical_failure_diagnostics(audit_df, conv)

# Observable ratios diagnose whether the signal is predominantly weak/single-phonon.
def _safe_ratio(num, den):
    num = pd.to_numeric(num, errors="coerce").to_numpy(float)
    den = pd.to_numeric(den, errors="coerce").to_numpy(float)
    out = np.full(num.shape, np.nan, dtype=float)
    good = np.isfinite(num) & np.isfinite(den) & (den > 0.0) & (num >= 0.0)
    out[good] = num[good] / den[good]
    return out

audit_df["ratio_ge1_to_phonon"] = _safe_ratio(audit_df["event_rate_ge1_s"], audit_df["phonon_rate_s"])
audit_df["ratio_exact_M_to_ge1"] = _safe_ratio(audit_df["event_rate_exact_M_s"], audit_df["event_rate_ge1_s"])

AUDIT_CSV = OUTPUT / "staged_event_rate_convergence_audit.csv"
CLASSIFIED_CSV = OUTPUT / "staged_event_rate_classified.csv"
audit_df.to_csv(AUDIT_CSV, index=False)
audit_df.to_csv(CLASSIFIED_CSV, index=False)
# Backward-compatible central-estimate filename. It contains valid points only.
audit_df.to_csv(OUTPUT / "staged_event_rate.csv", index=False)

# Strict precision-certified table retained for backward compatibility.
reliable_df = audit_df.copy()
metric_flags = {
    "phonon_rate_s": "reliable_phonon",
    "event_rate_ge1_s": "reliable_ge1",
    "event_rate_exact_M_s": "reliable_exact_M",
}
for metric, flag in metric_flags.items():
    reliable_df.loc[~reliable_df[flag].astype(bool), metric] = np.nan
RELIABLE_CSV = OUTPUT / "staged_event_rate_reliable.csv"
reliable_df.to_csv(RELIABLE_CSV, index=False)

# Six-level classification summary for each observable.
classification_rows = []
for observable, label in [
    ("Ndot_ph", "phonon"),
    ("Gamma_ge1", "ge1"),
    (f"Gamma_M{cfg.exact_phonon_number}", "exact_M"),
]:
    counts = audit_df[f"{label}_classification"].value_counts()
    for code in range(5, -1, -1):
        cls = RESULT_CLASS_LABELS[code]
        classification_rows.append({
            "observable": observable,
            "classification": cls,
            "points": int(counts.get(cls, 0)),
            "valid_points": _expected_n,
            "fraction": float(counts.get(cls, 0) / _expected_n) if _expected_n else np.nan,
        })
classification_summary = pd.DataFrame(classification_rows)
classification_summary.to_csv(OUTPUT / "staged_classification_summary.csv", index=False)
classification_wide = (classification_summary
    .pivot(index="observable", columns="classification", values="points")
    .fillna(0).astype(int)
    .reindex(columns=[RESULT_CLASS_LABELS[i] for i in range(5, -1, -1)], fill_value=0))
classification_wide["valid_points"] = _expected_n
classification_wide.to_csv(OUTPUT / "staged_classification_summary_wide.csv")
print("Six-level quality distribution (this is the table to use for estimated/noisy/unresolved counts):")
display(classification_wide)

upgrade_candidates = {
    "phonon": int(audit_df.get("phonon_needs_secondary_audit", pd.Series(False, index=audit_df.index)).sum()),
    "Gamma_ge1": int(audit_df.get("ge1_needs_secondary_audit", pd.Series(False, index=audit_df.index)).sum()),
    f"Gamma_M{cfg.exact_phonon_number}": int(audit_df.get("exact_M_needs_secondary_audit", pd.Series(False, index=audit_df.index)).sum()),
}
print("Old checkpoint rows that could potentially be upgraded from estimated to well_converged after a secondary audit:")
print(upgrade_candidates)

# Keep the old precision-certification summary as a compact reference.
summary = pd.DataFrame({
    "observable": [r"Ndot_ph", r"Gamma_ge1", f"Gamma_M{cfg.exact_phonon_number}", "all_three"],
    "precision_certified_points": [
        int((audit_df.phonon_class_code == 5).sum()),
        int((audit_df.ge1_class_code == 5).sum()),
        int((audit_df.exact_M_class_code == 5).sum()),
        int((audit_df.overall_class_code == 5).sum()),
    ],
    "valid_points": _expected_n,
})
summary["fraction_precision_certified"] = summary["precision_certified_points"] / _expected_n
summary.to_csv(OUTPUT / "staged_reliability_summary.csv", index=False)
print("Strict 5% precision-certification summary. A zero here does NOT mean the points are unresolved; consult the six-level table above.")
display(summary)

print("Classification reasons (top 20 per observable):")
for _label in ("phonon", "ge1", "exact_M"):
    print(f"\n{_label}:")
    display(audit_df[f"{_label}_classification_reason"].value_counts(dropna=False).rename("count").to_frame().head(20))

# Flatten the adaptive-b history so the old/new annulus evidence can be inspected without rerunning trajectories.
TAIL_HISTORY_CSV = OUTPUT / "staged_adaptive_bmax_history.csv"
tail_history_df = flatten_adaptive_b_history(audit_df)
tail_history_df.to_csv(TAIL_HISTORY_CSV, index=False)
print("Adaptive-b history CSV:", TAIL_HISTORY_CSV, "rows=", len(tail_history_df))

print("Audit/classified CSV:", CLASSIFIED_CSV)
print("Precision-certified-only CSV:", RELIABLE_CSV)
print("Failure modes:")
display(audit_df["reliability_failures"].value_counts(dropna=False).rename("count").to_frame().head(20))
display(audit_df.head())


## 5. Central-rate, uncertainty, classification, and support diagnostics

Each observable should be interpreted together with its central rate, Monte Carlo uncertainty, and six-level quality classification. Version 2026-08-13.2 additionally produces:

- **adaptive-$b_{\max}$ support repair:** previous central estimates and previous annulus summaries are reused; statistically ambiguous tails receive more samples at fixed support before any geometric expansion;
- **support-status diagnostics:** `converged`, `tail_noise_limited`, `support_unresolved_max_factor`, or `geometry_limited`;
- **adaptive-$b$ history CSV:** every retained support factor, annulus sample size, mean/lower/upper tail fraction, and repair decision;
- **dominant numerical-failure maps:** `far_outer`, `timestep`, `tolerances`, `stage_time`, `switch`, `ground`, `pass_all`, or `not_audited`;
- **fixed precision-certified maps:** finite but non-certified points are shown separately from genuinely missing central estimates;
- **observable-ratio maps:** $\Gamma_{\ge1}/\dot N_{\rm ph}$ and $\Gamma_M/\Gamma_{\ge1}$. For $M=1$, ratios near unity indicate a predominantly weak, one-phonon excitation regime.

The checkpoint remains canonicalized to exactly one row per current pseudopotential-valid grid point. A black `x` is reserved only for a valid point with no finite central estimate.


In [ ]:
# -------------------------------------------------------------------------
# A. ACTUAL CENTRAL RATE MAPS - do not blank finite estimates just because
#    they miss the 5% precision-certification threshold.
# -------------------------------------------------------------------------
plot_metric(audit_df, cfg, "phonon_rate_s",
            "Staged selected-mode phonon production rate - central estimate",
            r"$\dot N_{\mathrm{ph},k}$ [s$^{-1}$]",
            OUTPUT / "staged_actual_phonon_rate.png")
plot_metric(audit_df, cfg, "event_rate_ge1_s",
            "Staged at-least-one-phonon event rate - central estimate",
            r"$\Gamma_{\geq1,k}$ [s$^{-1}$]",
            OUTPUT / "staged_actual_gamma_ge1.png")
plot_metric(audit_df, cfg, "event_rate_exact_M_s",
            f"Staged exact-{cfg.exact_phonon_number}-phonon event rate - central estimate",
            rf"$\Gamma_{{{cfg.exact_phonon_number},k}}$ [s$^{{-1}}$]",
            OUTPUT / f"staged_actual_gamma_M{cfg.exact_phonon_number}.png")
# Backward-compatible main heatmap now shows the actual central estimate.
plot_metric(audit_df, cfg, "phonon_rate_s",
            "Staged selected-mode phonon production rate - central estimate",
            r"$\dot N_{\mathrm{ph},k}$ [s$^{-1}$]",
            OUTPUT / "staged_heatmap.png")

# -------------------------------------------------------------------------
# B. MONTE CARLO UNCERTAINTY MAPS
# -------------------------------------------------------------------------
plot_relative_error(audit_df, cfg, "phonon_mc_rel_se",
                    "Phonon-rate Monte Carlo uncertainty",
                    OUTPUT / "staged_error_phonon_mc_rel_se.png",
                    cap_pct=100.0 * conv.noisy_rel_se)
plot_relative_error(audit_df, cfg, "ge1_mc_rel_se",
                    r"$\Gamma_{\geq1}$ Monte Carlo uncertainty",
                    OUTPUT / "staged_error_gamma_ge1_mc_rel_se.png",
                    cap_pct=100.0 * conv.noisy_rel_se)
plot_relative_error(audit_df, cfg, "exact_M_mc_rel_se",
                    rf"$\Gamma_{{{cfg.exact_phonon_number}}}$ Monte Carlo uncertainty",
                    OUTPUT / f"staged_error_gamma_M{cfg.exact_phonon_number}_mc_rel_se.png",
                    cap_pct=100.0 * conv.noisy_rel_se)

# -------------------------------------------------------------------------
# C. FIVE-LEVEL QUALITY CLASSIFICATION MAPS
# -------------------------------------------------------------------------
plot_classification(audit_df, cfg, "phonon_class_code",
                    "Quality classification - phonon production rate",
                    OUTPUT / "staged_classification_phonon.png")
plot_classification(audit_df, cfg, "ge1_class_code",
                    r"Quality classification - $\Gamma_{\geq1}$",
                    OUTPUT / "staged_classification_gamma_ge1.png")
plot_classification(audit_df, cfg, "exact_M_class_code",
                    rf"Quality classification - $\Gamma_{{{cfg.exact_phonon_number}}}$",
                    OUTPUT / f"staged_classification_gamma_M{cfg.exact_phonon_number}.png")

# -------------------------------------------------------------------------
# D. STRICT PRECISION-CERTIFIED MAPS. Finite non-certified points are gray;
#    black x markers mean genuinely missing central estimates only.
# -------------------------------------------------------------------------
plot_precision_certified_metric(audit_df, cfg, "phonon_rate_s", "phonon_class_code",
            "Precision-certified staged phonon production rate",
            r"$\dot N_{\mathrm{ph},k}$ [s$^{-1}$]",
            OUTPUT / "staged_reliable_phonon_rate.png")
plot_precision_certified_metric(audit_df, cfg, "event_rate_ge1_s", "ge1_class_code",
            "Precision-certified staged at-least-one-phonon event rate",
            r"$\Gamma_{\geq1,k}$ [s$^{-1}$]",
            OUTPUT / "staged_reliable_gamma_ge1.png")
plot_precision_certified_metric(audit_df, cfg, "event_rate_exact_M_s", "exact_M_class_code",
            f"Precision-certified staged exact-{cfg.exact_phonon_number}-phonon event rate",
            rf"$\Gamma_{{{cfg.exact_phonon_number},k}}$ [s$^{{-1}}$]",
            OUTPUT / f"staged_reliable_gamma_M{cfg.exact_phonon_number}.png")

# Additional detailed convergence diagnostics.
plot_convergence_quantity(audit_df, cfg, "b_tail_phonon_fraction_upper",
                          "Upper bound on omitted impact-parameter tail",
                          r"upper bound on $\Delta\dot N_{\rm tail}/\dot N$",
                          OUTPUT / "staged_convergence_btail_phonon.png",
                          vmax=max(0.20, conv.b_tail_fraction_tol * 3))
if "max_numerical_phonon_relative_upper" in audit_df:
    plot_convergence_quantity(audit_df, cfg, "max_numerical_phonon_relative_upper",
                              "Largest paired numerical-resolution shift",
                              "2-sigma relative upper bound",
                              OUTPUT / "staged_convergence_numerical_phonon.png",
                              vmax=max(0.20, conv.numerical_rel_tol * 3))


# Adaptive-support size itself: 1 means the original Rutherford B was enough;
# 2,4,8,... mean successively enlarged central support.
if "adaptive_bmax_factor" in audit_df:
    plot_convergence_quantity(audit_df, cfg, "adaptive_bmax_factor",
                              "Adaptive impact-parameter support multiplier",
                              r"final central support / nominal $B$",
                              OUTPUT / "staged_adaptive_bmax_factor.png",
                              vmax=max(4.0, float(pd.to_numeric(audit_df["adaptive_bmax_factor"], errors="coerce").max())))

plot_support_status(audit_df, cfg, OUTPUT / "staged_adaptive_bmax_status.png")
if "tail_history_df" in globals() and not tail_history_df.empty:
    plot_tail_history_representatives(tail_history_df, audit_df, OUTPUT / "staged_adaptive_bmax_tail_history_representatives.png")

# Dominant numerical-failure mode, separately for each observable.
plot_numerical_failure_mode(audit_df, cfg, "dominant_numerical_phonon_code",
                            "Dominant numerical convergence mode - phonon rate",
                            OUTPUT / "staged_numerical_failure_phonon.png")
plot_numerical_failure_mode(audit_df, cfg, "dominant_numerical_ge1_code",
                            r"Dominant numerical convergence mode - $\Gamma_{\geq1}$",
                            OUTPUT / "staged_numerical_failure_gamma_ge1.png")
plot_numerical_failure_mode(audit_df, cfg, "dominant_numerical_exact_M_code",
                            rf"Dominant numerical convergence mode - $\Gamma_{{{cfg.exact_phonon_number}}}$",
                            OUTPUT / f"staged_numerical_failure_gamma_M{cfg.exact_phonon_number}.png")

# Observable-ratio maps.  Both ratios are bounded by 1 for the coherent-state
# observables used here; values near 1 diagnose weak/single-phonon excitation.
plot_observable_ratio(audit_df, cfg, "event_rate_ge1_s", "phonon_rate_s",
                      r"Observable ratio $\Gamma_{\geq1}/\dot N_{\rm ph}$",
                      r"$\Gamma_{\geq1}/\dot N_{\rm ph}$",
                      OUTPUT / "staged_ratio_gamma_ge1_to_phonon.png")
plot_observable_ratio(audit_df, cfg, "event_rate_exact_M_s", "event_rate_ge1_s",
                      rf"Observable ratio $\Gamma_{{{cfg.exact_phonon_number}}}/\Gamma_{{\geq1}}$",
                      rf"$\Gamma_{{{cfg.exact_phonon_number}}}/\Gamma_{{\geq1}}$",
                      OUTPUT / f"staged_ratio_gamma_M{cfg.exact_phonon_number}_to_ge1.png")


## 6. Staged / screening comparison

The comparison CSV now uses the **actual staged central estimates** rather than deleting every point that misses 5% precision. Interpret the ratio together with the staged uncertainty and classification maps. The strict precision-certified-only table remains available separately.


In [ ]:
def plot_ratio_map(comparison, ratio_column, label, title, path):
    common = comparison.dropna(subset=[ratio_column]).copy()
    common = common[np.isfinite(common[ratio_column]) & (common[ratio_column] > 0)]
    if common.empty:
        print("No finite common points for", ratio_column)
        return
    masses = np.sort(common.m_dm_kg.unique())
    eps = np.sort(common.eps.unique())
    R = (common.pivot(index="eps", columns="m_dm_kg", values=ratio_column)
         .reindex(index=eps, columns=masses).to_numpy().T)
    logR = np.log10(np.where(R > 0, R, np.nan))
    vmax = max(0.5, float(np.nanmax(np.abs(logR)))) if np.isfinite(logR).any() else 1.0
    fig, ax = plt.subplots(figsize=(9.2, 6.5))
    _draw_reference_background(ax, cfg)
    im = ax.pcolormesh(_log_edges(eps), _log_edges(masses), logR, shading="flat",
                       cmap="coolwarm", norm=TwoSlopeNorm(vcenter=0, vmin=-vmax, vmax=vmax), zorder=3)
    fig.colorbar(im, ax=ax).set_label(label)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlim(cfg.eps_min, cfg.eps_max); ax.set_ylim(cfg.m_min_kg, cfg.m_max_kg)
    ax.set_xlabel(r"Charge fraction $\epsilon$ ($Q=\epsilon e$)")
    ax.set_ylabel(r"DM mass $m_\chi$ [kg]")
    ax.set_title(title)
    fig.tight_layout(); fig.savefig(path, dpi=180); plt.show()

screen_path = OUTPUT / "screening_event_rate.csv"
if screen_path.exists():
    screening_df = pd.read_csv(screen_path)
    comparison = compare_frames(audit_df, screening_df)
    comparison.to_csv(OUTPUT / "staged_vs_screening.csv", index=False)
    plot_ratio_map(comparison, "phonon_rate_s_ratio_staged_over_screening",
                   r"$\log_{10}(\dot N_{\rm staged}/\dot N_{\rm screen})$",
                   "Staged / screening phonon-rate ratio",
                   OUTPUT / "staged_vs_screening_phonon_ratio.png")
    plot_ratio_map(comparison, "event_rate_ge1_s_ratio_staged_over_screening",
                   r"$\log_{10}(\Gamma_{\geq1}^{\rm staged}/\Gamma_{\geq1}^{\rm screen})$",
                   r"Staged / screening $\Gamma_{\geq1}$ ratio",
                   OUTPUT / "staged_vs_screening_gamma_ge1_ratio.png")
    plot_ratio_map(comparison, "event_rate_exact_M_s_ratio_staged_over_screening",
                   rf"$\log_{{10}}(\Gamma_{{{cfg.exact_phonon_number}}}^{{\rm staged}}/\Gamma_{{{cfg.exact_phonon_number}}}^{{\rm screen}})$",
                   rf"Staged / screening $\Gamma_{{{cfg.exact_phonon_number}}}$ ratio",
                   OUTPUT / f"staged_vs_screening_gamma_M{cfg.exact_phonon_number}_ratio.png")
else:
    print("Run the screening notebook first.")


## 7. How to interpret the six quality classes

The central rate, the MC error, support status, and classification are separate pieces of information. `precision_certified` is the strict convergence claim; `well_converged` is a strong quantitative result with up to 10% MC error and passed support/numerical audits; `estimated` has bounded impact-parameter support and <=20% MC error but is not fully numerically certified; `support_unresolved` has a finite central estimate but the outer-$b$ integral is not yet bounded; `noisy` has bounded support but 20--50% MC error; and `unresolved` should not be used as a numerical prediction.

A failed $b$-support audit no longer produces an ordinary `estimated` label. It produces `support_unresolved`, because omitted phase space is qualitatively different from sampling noise. Numerical audits are intentionally blocked until the support audit passes. A bounded-support point can still be `estimated` when its MC error is <=20% but the numerical audit is absent or incomplete. The `*_classification_reason`, `reliability_failures`, `b_tail_*`, and numerical-audit columns in `staged_event_rate_classified.csv` record the reason.

These quality classes describe **numerical/statistical confidence in the stated staged model**. They do not remove the separate systematic uncertainty associated with the phenomenological copper-transport model.
